<a href="https://colab.research.google.com/github/prathibhaogirala-creator/Pallavi-Ogirala/blob/main/Artifacts12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
<!DOCTYPE html>
<html>
<head>
  <title>Cayley-Hamilton Verifier</title>

  <style>
    body {
      font-family: Arial, sans-serif;
      background: #f4f6f8;
      margin: 0;
      padding: 30px;
      text-align: center;
    }

    .container {
      max-width: 800px;
      margin: auto;
      background: white;
      padding: 25px;
      border-radius: 15px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }

    h1 {
      margin-bottom: 5px;
    }

    .matrix-input {
      display: grid;
      gap: 10px;
      justify-content: center;
      margin: 25px;
    }

    input {
      width: 60px;
      padding: 10px;
      text-align: center;
      font-size: 16px;
    }

    button {
      padding: 12px 25px;
      font-size: 16px;
      border: none;
      border-radius: 8px;
      cursor: pointer;
      background: #222;
      color: white;
      margin: 5px;
    }

    button:hover {
      opacity: 0.85;
    }

    #result {
      margin-top: 20px;
      padding: 15px;
      border-radius: 10px;
      background: #f0f0f0;
      text-align: left;
    }

    .success {
      color: green;
      font-weight: bold;
    }

    .error {
      color: red;
      font-weight: bold;
    }
  </style>
</head>

<body>

<div class="container">

  <h1>Cayley-Hamilton Verifier Tool</h1>
  <p>Verify whether a matrix satisfies its characteristic equation.</p>

  <label>Choose Matrix Size:</label>

  <select id="size" onchange="createMatrix()">
    <option value="2">2 × 2</option>
    <option value="3">3 × 3</option>
  </select>

  <div id="matrix" class="matrix-input"></div>

  <button onclick="verifyMatrix()">Verify Cayley-Hamilton</button>
  <button onclick="clearMatrix()">Clear</button>

  <div id="result">
    Enter a matrix and click Verify.
  </div>

</div>

<script>

function createMatrix() {

  const n = Number(document.getElementById("size").value);
  const matrixDiv = document.getElementById("matrix");

  matrixDiv.innerHTML = "";

  matrixDiv.style.gridTemplateColumns =
    `repeat(${n}, 70px)`;

  for (let i = 0; i < n; i++) {

    for (let j = 0; j < n; j++) {

      const input = document.createElement("input");

      input.type = "number";
      input.value = "0";
      input.id = `a${i}${j}`;

      matrixDiv.appendChild(input);
    }
  }
}

function getMatrix() {

  const n = Number(document.getElementById("size").value);

  let matrix = [];

  for (let i = 0; i < n; i++) {

    let row = [];

    for (let j = 0; j < n; j++) {

      row.push(
        Number(document.getElementById(`a${i}${j}`).value)
      );

    }

    matrix.push(row);
  }

  return matrix;
}

async function verifyMatrix() {

  const matrix = getMatrix();

  try {

    const response = await fetch("/verify", {

      method: "POST",

      headers: {
        "Content-Type": "application/json"
      },

      body: JSON.stringify({
        matrix: matrix
      })

    });

    const data = await response.json();

    document.getElementById("result").innerHTML =

      `<h3>Result</h3>
       <p><b>Characteristic Polynomial:</b>
       ${data.characteristic_polynomial}</p>

       <p><b>p(A):</b></p>
       <pre>${JSON.stringify(data.pA, null, 2)}</pre>

       <p class="${data.verified ? 'success' : 'error'}">
       ${data.message}
       </p>`;

  } catch (error) {

    document.getElementById("result").innerHTML =
      `<p class="error">
      Backend connection error.
      </p>`;

  }
}

function clearMatrix() {

  createMatrix();

  document.getElementById("result").innerHTML =
    "Enter a matrix and click Verify.";
}

createMatrix();

</script>

</body>
</html>

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import sympy as sp

app = FastAPI(
    title="Cayley-Hamilton Verifier API"
)


class MatrixRequest(BaseModel):
    matrix: list[list[float]]


def verify_cayley_hamilton(matrix):

    A = sp.Matrix(matrix)

    n = A.rows

    if A.cols != n:
        raise ValueError("Matrix must be square.")

    if n not in [2, 3]:
        raise ValueError(
            "Only 2x2 and 3x3 matrices are supported."
        )

    # Variable for characteristic polynomial
    x = sp.symbols("x")

    # Characteristic polynomial
    char_poly = A.charpoly(x)

    polynomial = char_poly.as_expr()

    # Evaluate characteristic polynomial at A
    coeffs = char_poly.all_coeffs()

    pA = sp.zeros(n)

    for i, coefficient in enumerate(coeffs):

        power = n - i

        if power == 0:
            term = coefficient * sp.eye(n)

        else:
            term = coefficient * (A ** power)

        pA = pA + term

    # Simplify matrix
    pA = pA.applyfunc(sp.simplify)

    # Check whether p(A) = zero matrix
    verified = all(
        element == 0
        for element in pA
    )

    return {
        "characteristic_polynomial":
            str(polynomial),

        "pA":
            pA.tolist(),

        "verified":
            verified,

        "message":
            "Cayley-Hamilton theorem verified successfully!"
            if verified
            else
            "Cayley-Hamilton theorem is not satisfied."
    }


@app.post("/verify")
def verify(request: MatrixRequest):

    return verify_cayley_hamilton(
        request.matrix
    )


@app.get("/")
def home():

    return {
        "message":
        "Cayley-Hamilton Verifier Tool is running."
    }